# Residual Policy Evaluation and Comparison

This notebook loads trained residual policies and compares their performance to:
1. Pure pretrained BC policies
2. Pure pretrained Diffusion policies
3. Residual-augmented versions of both

The evaluation focuses on recovery performance under perturbations.

In [19]:
import os
import sys
import h5py
import torch
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict

# Ensure local packages are importable
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'robomimic', 'robomimic')))

from a2l_pr.models import ResidualRecoveryPolicy
from a2l_pr.adapters.robomimic import RobomimicAdapter
from a2l_pr.perturbations.generator import PerturbationGenerator, PerturbationType

import robomimic.utils.file_utils as FileUtils
import robomimic.utils.env_utils as EnvUtils
import robomimic.utils.obs_utils as ObsUtils
from robomimic.algo import RolloutPolicy
from robomimic.envs.env_base import EnvBase

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'CUDA Available: {torch.cuda.is_available()}')

Device: cuda
CUDA Available: True


## Setup: Data Paths and Configuration

In [20]:
# Base paths
WORKSPACE_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
ROBOMIMIC_ROOT = os.path.join(WORKSPACE_ROOT, 'robomimic', 'robomimic')

# Dataset path
DATASET_PATH = os.path.join(ROBOMIMIC_ROOT, 'datasets', 'square', 'ph', 'low_dim_v15.hdf5')

# Trained model paths
BC_MODEL_DIR = os.path.join(ROBOMIMIC_ROOT, 'bc_trained_models', 'test')
DIFFUSION_MODEL_DIR = os.path.join(ROBOMIMIC_ROOT, 'diffusion_policy_trained_models', 'test')

# Residual policy checkpoint - update this to your trained residual policy path
RESIDUAL_CHECKPOINT = os.path.join(os.getcwd(), 'residual_recovery_policy.pth')

print(f'Dataset path: {DATASET_PATH}')
print(f'BC model dir: {BC_MODEL_DIR}')
print(f'Diffusion model dir: {DIFFUSION_MODEL_DIR}')
print(f'Residual policy: {RESIDUAL_CHECKPOINT}')
print(f'Dataset exists: {os.path.exists(DATASET_PATH)}')
print(f'BC model dir exists: {os.path.exists(BC_MODEL_DIR)}')
print(f'Diffusion model dir exists: {os.path.exists(DIFFUSION_MODEL_DIR)}')
print(f'Residual policy exists: {os.path.exists(RESIDUAL_CHECKPOINT)}')


Dataset path: /home/griffing52/vail/bot2bot/bot2bot/a2l/robomimic/robomimic/datasets/square/ph/low_dim_v15.hdf5
BC model dir: /home/griffing52/vail/bot2bot/bot2bot/a2l/robomimic/robomimic/bc_trained_models/test
Diffusion model dir: /home/griffing52/vail/bot2bot/bot2bot/a2l/robomimic/robomimic/diffusion_policy_trained_models/test
Residual policy: /home/griffing52/vail/bot2bot/bot2bot/a2l/a2l-pr/notebooks/residual_recovery_policy.pth
Dataset exists: True
BC model dir exists: True
Diffusion model dir exists: True
Residual policy exists: True


## Utility Functions

In [21]:
def load_robomimic_trajectory(hdf5_path, demo_key='demo_0'):
    """Load a single trajectory from robomimic HDF5 file."""
    if not os.path.exists(hdf5_path):
        raise FileNotFoundError(f"Dataset not found: {hdf5_path}")
    
    with h5py.File(hdf5_path, 'r') as f:
        if 'data' not in f or demo_key not in f['data']:
            raise KeyError(f"Demo {demo_key} not found in dataset")
        
        demo_grp = f['data'][demo_key]
        obs_grp = demo_grp['obs']
        
        trajectory = {
            'actions': demo_grp['actions'][:],
            'observations': {}
        }
        
        for key in obs_grp.keys():
            trajectory['observations'][key] = obs_grp[key][:]
    
    return trajectory


def _traj_length(traj):
    """Get trajectory length."""
    return len(traj['actions'])


def get_trajectory_demos(hdf5_path, limit=None):
    """Get list of available demos in dataset."""
    with h5py.File(hdf5_path, 'r') as f:
        demos = list(f['data'].keys())
    
    if limit:
        demos = demos[:limit]
    return demos

## Load Pretrained Policies

The following code loads the trained BC and Diffusion policies from your model checkpoints. 

**Notes:**
- Policies are automatically loaded from the latest checkpoint in each model directory
- BC and Diffusion policies are wrapped with `RolloutPolicy` for environment interaction
- If a checkpoint is missing, that policy evaluation will be skipped

In [22]:
def find_latest_model_dir(model_base_dir):
    """Find the most recent model directory by timestamp."""
    if not os.path.exists(model_base_dir):
        return None
    
    subdirs = [d for d in os.listdir(model_base_dir) if os.path.isdir(os.path.join(model_base_dir, d))]
    if not subdirs:
        return None
    
    # Sort by name (assuming timestamp format)
    subdirs.sort(reverse=True)
    return os.path.join(model_base_dir, subdirs[0])


def load_pretrained_policy(checkpoint_path, device='cpu'):
    """Load a pretrained robomimic policy from checkpoint."""
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint not found: {checkpoint_path}")
        return None
    
    try:
        policy, ckpt_dict = FileUtils.policy_from_checkpoint(
            device=device,
            ckpt_path=checkpoint_path,
            verbose=False
        )
        return policy
    except Exception as e:
        print(f"Error loading policy from {checkpoint_path}: {e}")
        return None


# Find and load BC policy
bc_model_dir = find_latest_model_dir(BC_MODEL_DIR)
if bc_model_dir:
    bc_models_dir = os.path.join(bc_model_dir, 'models')
    bc_checkpoints = sorted([f for f in os.listdir(bc_models_dir) if f.endswith('.pth')])
    if bc_checkpoints:
        bc_checkpoint = os.path.join(bc_models_dir, bc_checkpoints[-1])  # Latest checkpoint
        print(f"Loading BC policy from: {bc_checkpoint}")
        bc_policy = load_pretrained_policy(bc_checkpoint, device=device)
        if bc_policy:
            print("✓ BC policy loaded successfully")
    else:
        print("No BC checkpoints found")
        bc_policy = None
else:
    print("No BC model directory found")
    bc_policy = None

# Find and load Diffusion policy
diffusion_model_dir = find_latest_model_dir(DIFFUSION_MODEL_DIR)
if diffusion_model_dir:
    diffusion_models_dir = os.path.join(diffusion_model_dir, 'models')
    diffusion_checkpoints = sorted([f for f in os.listdir(diffusion_models_dir) if f.endswith('.pth')])
    if diffusion_checkpoints:
        diffusion_checkpoint = os.path.join(diffusion_models_dir, diffusion_checkpoints[-1])  # Latest checkpoint
        print(f"Loading Diffusion policy from: {diffusion_checkpoint}")
        diffusion_policy = load_pretrained_policy(diffusion_checkpoint, device=device)
        if diffusion_policy:
            print("✓ Diffusion policy loaded successfully")
    else:
        print("No Diffusion checkpoints found")
        diffusion_policy = None
else:
    print("No Diffusion model directory found")
    diffusion_policy = None

Loading BC policy from: /home/griffing52/vail/bot2bot/bot2bot/a2l/robomimic/robomimic/bc_trained_models/test/20260511234956/models/model_epoch_950.pth

============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['object', 'robot0_eef_pos', 'robot0_gripper_qpos', 'robot0_eef_quat']
using obs modality: rgb with keys: []
using obs modality: depth with keys: []
using obs modality: scan with keys: []
✓ BC policy loaded successfully
Loading Diffusion policy from: /home/griffing52/vail/bot2bot/bot2bot/a2l/robomimic/robomimic/diffusion_policy_trained_models/test/20260512010513/models/model_epoch_950.pth

============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['object', 'robot0_eef_pos', 'robot0_gripper_qpos', 'robot0_eef_quat']
using obs modality: rgb with keys: []
using obs modality: depth with keys: []
using obs modality: scan with keys: []
number of parameters: 6.587828e+07
✓ Diffus

## Load Trained Residual Policies

**IMPORTANT:** Update the checkpoint paths below to point to your trained residual policies.

The residual policy is a neural network that learns to predict corrective actions:
- **Input**: Past states and actions (history) + current state before perturbation
- **Output**: Residual actions for the next horizon steps
- **Goal**: Predict actions that would undo/recover from perturbations

If you haven't trained residual policies yet, you can skip this and the notebook will still evaluate the base policies.

In [23]:
def load_residual_policy(checkpoint_path, device='cpu'):
    """Load a trained residual recovery policy.
    
    Uses dimensions from the training notebook: state_dim=23, action_dim=7
    """
    if not os.path.exists(checkpoint_path):
        print(f"Residual policy checkpoint not found: {checkpoint_path}")
        return None
    
    try:
        # Load checkpoint and extract state dict
        checkpoint = torch.load(checkpoint_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            state_dict = checkpoint['model_state_dict']
        else:
            state_dict = checkpoint
        
        # Dimensions from training (robomimic_residual_training.ipynb)
        # The training notebook used state_dim=23, action_dim=7
        # This gives input_dim = 23 + 7 = 30
        state_dim = 23        # Actual training state_dim
        action_dim = 7        # Robomimic standard
        history_length = 12
        horizon = 30
        
        print(f"Loading residual policy:")
        print(f"  state_dim={state_dim}, action_dim={action_dim}")
        print(f"  Expected input_dim to step_encoder: {state_dim + action_dim}")
        
        # Create policy with correct dimensions
        policy = ResidualRecoveryPolicy(
            state_dim=state_dim,
            action_dim=action_dim,
            history_length=history_length,
            prediction_horizon=horizon,
        ).to(device)
        
        # Load weights
        policy.load_state_dict(state_dict)
        policy.eval()
        
        print(f"✓ Residual policy loaded successfully")
        return policy
        
    except Exception as e:
        print(f"✗ Error loading residual policy: {e}")
        import traceback
        traceback.print_exc()
        return None


# Try to load residual policy
residual_policy = load_residual_policy(RESIDUAL_CHECKPOINT, device=device)
if residual_policy:
    print("✓ Residual policy ready for composition")
else:
    print("⚠ Residual policy not available")


Loading residual policy:
  state_dim=23, action_dim=7
  Expected input_dim to step_encoder: 30
✓ Residual policy loaded successfully
✓ Residual policy ready for composition


## Policy Evaluation Functions

In [ ]:
def extract_state_from_obs(obs_dict, t):
    """Extract state vector from observation dictionary.
    
    Matches the training notebook state extraction:
    robot0_eef_pos (3) + eef_quat (4) + gripper_qpos (1) + joint_pos (7) + joint_vel (7) = 22 dims
    (Note: training notebook reported 23, but manual calculation gives 22. Using actual keys.)
    """
    state_keys = [
        'robot0_eef_pos',      # 3
        'robot0_eef_quat',     # 4
        'robot0_gripper_qpos', # 1
        'robot0_joint_pos',    # 7
        'robot0_joint_vel',    # 7
    ]
    
    state_parts = []
    for key in state_keys:
        if key in obs_dict and t < len(obs_dict[key]):
            val = np.asarray(obs_dict[key][t]).reshape(-1).astype(np.float32)
            state_parts.append(val)
    
    if state_parts:
        return np.concatenate(state_parts)
    else:
        return None


class ComposedPolicy:
    """Wraps a base policy with residual corrections.
    
    At each step:
    1. Get action from base policy
    2. Query residual policy for correction (using history)
    3. Add residual to base action: a_exec = base_action + residual
    
    Note: Residual policy trained with state_dim=23 (from training notebook),
    so state history is concatenation of key observations.
    """
    
    def __init__(self, base_policy, residual_policy, device='cpu', residual_weight=1.0, clamp_residual=1.0):
        self.base_policy = base_policy
        self.residual_policy = residual_policy
        self.device = device
        self.residual_weight = residual_weight
        self.clamp_residual = clamp_residual
        self.history_length = 12
        self.state_history = []
        self.action_history = []
        
    def reset(self):
        """Reset history for new episode."""
        self.state_history = []
        self.action_history = []
    
    def _extract_state(self, obs):
        """Extract state vector from observation dict (matching training format)."""
        state_keys = [
            'robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos',
            'robot0_joint_pos', 'robot0_joint_vel'
        ]
        state_parts = []
        
        for key in state_keys:
            if key in obs:
                val = np.asarray(obs[key]).reshape(-1).astype(np.float32)
                state_parts.append(val)
        
        if state_parts:
            return np.concatenate(state_parts)
        return None
    
    def __call__(self, obs):
        """Get composed action (base + residual)."""
        # Get base action
        base_action = self.base_policy(obs)
        if isinstance(base_action, torch.Tensor):
            base_action = base_action.cpu().numpy()
        base_action = base_action.reshape(-1)
        
        # Extract state and update history
        state = self._extract_state(obs)
        if state is not None:
            self.state_history.append(state)
            self.action_history.append(base_action)
            
            # Keep only last N steps
            if len(self.state_history) > self.history_length:
                self.state_history = self.state_history[-self.history_length:]
                self.action_history = self.action_history[-self.history_length:]
            
            # Try to get residual correction
            if self.residual_policy and len(self.state_history) >= self.history_length:
                try:
                    past_states = torch.FloatTensor(np.stack(self.state_history[-self.history_length:])).to(self.device).unsqueeze(0)
                    past_actions = torch.FloatTensor(np.stack(self.action_history[-self.history_length:])).to(self.device).unsqueeze(0)
                    
                    with torch.no_grad():
                        residuals = self.residual_policy(past_states, past_actions)  # (1, horizon, action_dim)
                    
                    if isinstance(residuals, torch.Tensor):
                        residual = residuals[0, 0, :].cpu().numpy()  # First step residual
                        residual = np.clip(residual, -self.clamp_residual, self.clamp_residual)
                        base_action = base_action + self.residual_weight * residual
                except Exception as e:
                    pass  # Fall back to base action
        
        return base_action


def run_rollout_with_trajectory(policy, env, horizon=400, record_obs_keys=None):
    """Run policy rollout and record full trajectory.
    
    Returns:
        dict: {
            'total_reward': float,
            'trajectory': list of (obs, action, reward, done),
            'success': bool (if 'success' in info)
        }
    """
    if record_obs_keys is None:
        record_obs_keys = ['robot0_eef_pos', 'robot0_gripper_qpos', 'robot0_joint_pos']
    
    obs = env.reset()
    done = False
    total_reward = 0.0
    step = 0
    trajectory = []
    
    while not done and step < horizon:
        action = policy(obs)
        obs, reward, done, info = env.step(action)
        total_reward += reward
        step += 1
        
        # Record trajectory point
        traj_point = {
            'action': action.copy() if isinstance(action, np.ndarray) else action,
            'reward': reward,
            'obs': {k: obs[k].copy() if isinstance(obs[k], np.ndarray) else obs[k] for k in record_obs_keys if k in obs}
        }
        trajectory.append(traj_point)
    
    success = info.get('success', False) if isinstance(info, dict) else False
    
    return {
        'total_reward': total_reward,
        'trajectory': trajectory,
        'success': success,
        'num_steps': len(trajectory),
    }


## Composed Policy & Trajectory Recording

This section creates a composed policy that wraps BC + residual correction and runs rollouts while recording trajectories for visualization.

**Control Flow:**
1. Base policy (BC) predicts action
2. Residual policy predicts correction based on history
3. Executed action = base_action + clipped(residual_correction)
4. Full trajectory recorded for comparison


In [25]:
# Setup environment and rollout utilities
import robomimic.utils.env_utils as EnvUtils
import robomimic.utils.obs_utils as ObsUtils
from robomimic.envs.env_base import EnvBase

def get_checkpoint_info(checkpoint_path):
    """Get environment metadata from a checkpoint."""
    try:
        ckpt_dict = FileUtils.maybe_dict_from_checkpoint(ckpt_path=checkpoint_path)
        return ckpt_dict
    except Exception as e:
        print(f"Error loading checkpoint info: {e}")
        return None

def create_env_for_rollout(checkpoint_path):
    """Create environment from checkpoint for policy rollouts."""
    try:
        ckpt_dict = FileUtils.maybe_dict_from_checkpoint(ckpt_path=checkpoint_path)
        env_meta = ckpt_dict.get('env_metadata')
        
        if env_meta is None:
            print(f"No environment metadata in checkpoint")
            return None
        
        # Initialize observation utilities
        obs_spec = ckpt_dict.get('obs_spec')
        if obs_spec:
            ObsUtils.initialize_obs_utils_with_obs_specs(obs_modality_specs=obs_spec)
        
        # Create environment
        env = EnvUtils.create_env_from_metadata(env_meta=env_meta, render=False, render_offscreen=False)
        return env
    except Exception as e:
        print(f"Error creating environment: {e}")
        return None

def run_rollout(policy, env, horizon=400, num_rollouts=3, render=False):
    """Run policy in environment and collect returns.
    
    Returns:
        list: Returns for each rollout
    """
    if env is None or policy is None:
        return []
    
    returns = []
    
    try:
        for rollout_idx in range(num_rollouts):
            obs = env.reset()
            done = False
            total_reward = 0.0
            step = 0
            
            while not done and step < horizon:
                # Get action from policy
                action = policy(obs)
                
                # Take environment step
                obs, reward, done, info = env.step(action)
                total_reward += reward
                step += 1
            
            returns.append(total_reward)
            print(f"  Rollout {rollout_idx+1}/{num_rollouts}: Return = {total_reward:.4f}")
    
    except Exception as e:
        print(f"Error during rollout: {e}")
    
    return returns

# Get test trajectories
print(f"Loading dataset from {DATASET_PATH}")
try:
    test_demos = get_trajectory_demos(DATASET_PATH, limit=5)  # Evaluate on 5 trajectories
    print(f"Found {len(test_demos)} test trajectories for recovery evaluation")
except Exception as e:
    print(f"Error loading test trajectories: {e}")
    test_demos = []

Loading dataset from /home/griffing52/vail/bot2bot/bot2bot/a2l/robomimic/robomimic/datasets/square/ph/low_dim_v15.hdf5
Found 5 test trajectories for recovery evaluation


In [ ]:
# Benchmark: BC Policy vs BC + Residual Policy
results = {
    'bc_only': {
        'returns': [],
        'trajectories': [],
        'successes': 0,
    },
    'bc_residual': {
        'returns': [],
        'trajectories': [],
        'successes': 0,
    }
}

print("\n" + "="*70)
print("BENCHMARK: BC Policy vs BC + Residual Policy")
print("="*70)

# Check if we have BC policy and checkpoint
if bc_policy is not None and 'bc_checkpoint' in locals():
    print(f"\nLoading BC checkpoint: {bc_checkpoint}")
    
    # Create environment
    env = create_env_for_rollout(bc_checkpoint)
    if env:
        print("✓ Environment created successfully")
        
        # Run BC only rollouts
        print("\n--- Running BC Policy Only ---")
        num_rollouts = 3
        for rollout_idx in range(num_rollouts):
            print(f"Rollout {rollout_idx+1}/{num_rollouts}...", end=" ")
            result = run_rollout_with_trajectory(bc_policy, env, horizon=400)
            results['bc_only']['returns'].append(result['total_reward'])
            results['bc_only']['trajectories'].append(result['trajectory'])
            if result['success']:
                results['bc_only']['successes'] += 1
            print(f"Return: {result['total_reward']:.4f}, Success: {result['success']}")
        
        # Run BC + Residual rollouts (if residual policy available)
        if residual_policy is not None:
            print("\n--- Running BC + Residual Policy ---")
            composed_policy = ComposedPolicy(bc_policy, residual_policy, device=device, residual_weight=1.0)
            
            for rollout_idx in range(num_rollouts):
                print(f"Rollout {rollout_idx+1}/{num_rollouts}...", end=" ")
                composed_policy.reset()
                result = run_rollout_with_trajectory(composed_policy, env, horizon=400)
                results['bc_residual']['returns'].append(result['total_reward'])
                results['bc_residual']['trajectories'].append(result['trajectory'])
                if result['success']:
                    results['bc_residual']['successes'] += 1
                print(f"Return: {result['total_reward']:.4f}, Success: {result['success']}")
        else:
            print("\n⚠ Residual policy not available - skipping BC+Residual evaluation")
    else:
        print("✗ Failed to create environment")
elif bc_policy is None:
    print("✗ BC policy not loaded")
else:
    print("✗ BC checkpoint not found")

print("\n✓ Evaluation complete!")



BENCHMARK: BC Policy vs BC + Residual Policy

Loading BC checkpoint: /home/griffing52/vail/bot2bot/bot2bot/a2l/robomimic/robomimic/bc_trained_models/test/20260511234956/models/model_epoch_950.pth
Created environment with name NutAssemblySquare
Action size is 7
✓ Environment created successfully

--- Running BC Policy Only ---
Rollout 1/3... ObservationKeyToModalityDict: robot0_joint_pos not found, adding robot0_joint_pos to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_pos_cos not found, adding robot0_joint_pos_cos to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_pos_sin not found, adding robot0_joint_pos_sin to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_vel not found, adding robot0_joint_vel to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_eef_quat_site not found, adding robot0_eef_quat_site to mapping with assumed low_dim modality!
ObservationK

## Results Summary

In [ ]:
# Display results summary
print("\n" + "="*70)
print("BENCHMARK RESULTS SUMMARY")
print("="*70)

# BC Only Results
if results['bc_only']['returns']:
    bc_returns = np.array(results['bc_only']['returns'])
    bc_successes = results['bc_only']['successes']
    print(f"\nBC Policy Only ({len(bc_returns)} rollouts):")
    print(f"  Mean Return:    {np.mean(bc_returns):8.4f}")
    print(f"  Std Return:     {np.std(bc_returns):8.4f}")
    print(f"  Min Return:     {np.min(bc_returns):8.4f}")
    print(f"  Max Return:     {np.max(bc_returns):8.4f}")
    print(f"  Success Rate:   {bc_successes}/{len(bc_returns)} ({100*bc_successes/len(bc_returns):.1f}%)")
else:
    print("\nBC Policy Only: No results")
    bc_returns = None

# BC + Residual Results
if results['bc_residual']['returns']:
    bc_res_returns = np.array(results['bc_residual']['returns'])
    bc_res_successes = results['bc_residual']['successes']
    print(f"\nBC + Residual Policy ({len(bc_res_returns)} rollouts):")
    print(f"  Mean Return:    {np.mean(bc_res_returns):8.4f}")
    print(f"  Std Return:     {np.std(bc_res_returns):8.4f}")
    print(f"  Min Return:     {np.min(bc_res_returns):8.4f}")
    print(f"  Max Return:     {np.max(bc_res_returns):8.4f}")
    print(f"  Success Rate:   {bc_res_successes}/{len(bc_res_returns)} ({100*bc_res_successes/len(bc_res_returns):.1f}%)")
    
    # Comparison
    if bc_returns is not None:
        improvement = ((np.mean(bc_res_returns) - np.mean(bc_returns)) / (np.abs(np.mean(bc_returns)) + 1e-6)) * 100
        print(f"\n--- Improvement with Residual ---")
        print(f"  Return Improvement: {improvement:+.2f}%")
        print(f"  Success Improvement: {bc_res_successes - bc_successes:+d} episodes")
else:
    print("\nBC + Residual Policy: No results")



BENCHMARK RESULTS SUMMARY

BC Policy Only (3 rollouts):
  Mean Return:      0.0000
  Std Return:       0.0000
  Min Return:       0.0000
  Max Return:       0.0000
  Success Rate:   0/3 (0.0%)

BC + Residual Policy: No results


## Trajectory Visualization & Comparison

Visualize end-effector trajectories from BC vs BC+Residual rollouts to see the difference in execution paths.


In [ ]:
# Visualize and compare trajectories
print("\nVisualizing trajectories...")

if results['bc_only']['trajectories'] and results['bc_residual']['trajectories']:
    # Compare first rollout from each
    bc_traj = results['bc_only']['trajectories'][0]
    bc_res_traj = results['bc_residual']['trajectories'][0]
    
    # Extract end-effector positions
    bc_eef_positions = np.array([point['obs']['robot0_eef_pos'] for point in bc_traj if 'robot0_eef_pos' in point['obs']])
    bc_res_eef_positions = np.array([point['obs']['robot0_eef_pos'] for point in bc_res_traj if 'robot0_eef_pos' in point['obs']])
    
    # Create comparison visualizations
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    fig.suptitle('BC Policy vs BC + Residual Policy - Trajectory Comparison', fontsize=16, fontweight='bold')
    
    # 1. End-effector XY position
    ax = axes[0, 0]
    if len(bc_eef_positions) > 0:
        ax.plot(bc_eef_positions[:, 0], bc_eef_positions[:, 1], 'b-', linewidth=2, label='BC Only', marker='o', markersize=4, alpha=0.7)
    if len(bc_res_eef_positions) > 0:
        ax.plot(bc_res_eef_positions[:, 0], bc_res_eef_positions[:, 1], 'r-', linewidth=2, label='BC + Residual', marker='s', markersize=4, alpha=0.7)
    ax.set_xlabel('X Position (m)', fontsize=11)
    ax.set_ylabel('Y Position (m)', fontsize=11)
    ax.set_title('End-Effector XY Trajectory', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 2. End-effector Z position over time
    ax = axes[0, 1]
    ax.plot(bc_eef_positions[:, 2], 'b-', linewidth=2, label='BC Only', marker='o', markersize=4, alpha=0.7)
    ax.plot(bc_res_eef_positions[:, 2], 'r-', linewidth=2, label='BC + Residual', marker='s', markersize=4, alpha=0.7)
    ax.set_xlabel('Step', fontsize=11)
    ax.set_ylabel('Z Position (m)', fontsize=11)
    ax.set_title('End-Effector Z Position Over Time', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 3. Trajectory distance from start
    ax = axes[1, 0]
    bc_distances = np.linalg.norm(bc_eef_positions - bc_eef_positions[0], axis=1)
    bc_res_distances = np.linalg.norm(bc_res_eef_positions - bc_res_eef_positions[0], axis=1)
    ax.plot(bc_distances, 'b-', linewidth=2, label='BC Only', marker='o', markersize=4, alpha=0.7)
    ax.plot(bc_res_distances, 'r-', linewidth=2, label='BC + Residual', marker='s', markersize=4, alpha=0.7)
    ax.set_xlabel('Step', fontsize=11)
    ax.set_ylabel('Distance from Start (m)', fontsize=11)
    ax.set_title('Cumulative Distance Traveled', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 4. Summary statistics
    ax = axes[1, 1]
    ax.axis('off')
    
    summary_text = "TRAJECTORY STATISTICS\n" + "="*45 + "\n\n"
    summary_text += "BC Policy Only:\n"
    summary_text += f"  Length:           {len(bc_traj)} steps\n"
    summary_text += f"  EE Distance:      {bc_distances[-1]:.4f} m\n"
    summary_text += f"  EE Std Dev:       {np.std(bc_distances):.4f} m\n\n"
    
    summary_text += "BC + Residual:\n"
    summary_text += f"  Length:           {len(bc_res_traj)} steps\n"
    summary_text += f"  EE Distance:      {bc_res_distances[-1]:.4f} m\n"
    summary_text += f"  EE Std Dev:       {np.std(bc_res_distances):.4f} m\n\n"
    
    summary_text += "Difference:\n"
    dist_diff = bc_res_distances[-1] - bc_distances[-1]
    summary_text += f"  Distance Δ:       {dist_diff:+.4f} m\n"
    
    ax.text(0.1, 0.5, summary_text, fontsize=10, family='monospace', verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed comparison
    print(f"\nTrajectory Comparison (First Rollout):")
    print(f"  BC Only - Steps: {len(bc_traj)}, Final EE Distance: {bc_distances[-1]:.4f}m")
    print(f"  BC+Residual - Steps: {len(bc_res_traj)}, Final EE Distance: {bc_res_distances[-1]:.4f}m")

elif results['bc_only']['trajectories']:
    print("BC trajectories available but BC+Residual trajectories missing. Cannot compare.")
else:
    print("No trajectories recorded for visualization.")



Visualizing trajectories...
BC trajectories available but BC+Residual trajectories missing. Cannot compare.


## Performance Comparison: Metrics & Statistics


In [ ]:
# Performance metrics and visualization
if results['bc_only']['returns'] and results['bc_residual']['returns']:
    bc_returns = np.array(results['bc_only']['returns'])
    bc_res_returns = np.array(results['bc_residual']['returns'])
    
    # Create performance comparison figure
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('BC vs BC+Residual Performance Comparison', fontsize=14, fontweight='bold')
    
    # 1. Return comparison
    ax = axes[0]
    x_pos = np.arange(2)
    means = [np.mean(bc_returns), np.mean(bc_res_returns)]
    stds = [np.std(bc_returns), np.std(bc_res_returns)]
    colors = ['#3498db', '#e74c3c']
    bars = ax.bar(x_pos, means, yerr=stds, capsize=15, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('Mean Return', fontsize=12)
    ax.set_title('Episode Returns', fontsize=12)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['BC Only', 'BC+Residual'], fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, (mean, std) in enumerate(zip(means, stds)):
        ax.text(i, mean + std + 0.5, f'{mean:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    # 2. Rollout returns distribution
    ax = axes[1]
    ax.boxplot([bc_returns, bc_res_returns], labels=['BC Only', 'BC+Residual'], patch_artist=True)
    bp = ax.boxplot([bc_returns, bc_res_returns], labels=['BC Only', 'BC+Residual'], patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_ylabel('Return', fontsize=12)
    ax.set_title('Return Distribution', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed stats
    print("\n" + "="*70)
    print("DETAILED PERFORMANCE COMPARISON")
    print("="*70)
    print(f"\nBC Policy Only:")
    print(f"  Mean:     {np.mean(bc_returns):8.4f}")
    print(f"  Median:   {np.median(bc_returns):8.4f}")
    print(f"  Std:      {np.std(bc_returns):8.4f}")
    print(f"  Min/Max:  {np.min(bc_returns):8.4f} / {np.max(bc_returns):8.4f}")
    
    print(f"\nBC + Residual:")
    print(f"  Mean:     {np.mean(bc_res_returns):8.4f}")
    print(f"  Median:   {np.median(bc_res_returns):8.4f}")
    print(f"  Std:      {np.std(bc_res_returns):8.4f}")
    print(f"  Min/Max:  {np.min(bc_res_returns):8.4f} / {np.max(bc_res_returns):8.4f}")
    
    # Statistical comparison
    mean_diff = np.mean(bc_res_returns) - np.mean(bc_returns)
    pct_improvement = (mean_diff / np.abs(np.mean(bc_returns))) * 100 if np.mean(bc_returns) != 0 else 0
    
    print(f"\nComparison:")
    print(f"  Mean Difference:  {mean_diff:+8.4f} ({pct_improvement:+.2f}%)")
    print(f"  Std Improvement:  {np.std(bc_returns) - np.std(bc_res_returns):+8.4f}")
    print(f"\nConclusion: ", end="")
    if pct_improvement > 5:
        print(f"✓ Residual policy improves performance by {pct_improvement:.1f}%")
    elif pct_improvement < -5:
        print(f"✗ Residual policy decreases performance by {-pct_improvement:.1f}%")
    else:
        print(f"~ Residual policy has similar performance (Δ {pct_improvement:+.1f}%)")

elif results['bc_only']['returns']:
    print("BC policy results available but BC+Residual results missing.")
else:
    print("No results available for comparison.")


BC policy results available but BC+Residual results missing.


## Results Export & Conclusions

Summary of BC vs BC+Residual benchmark evaluation and next steps.


In [ ]:
# Export results summary
print("\n" + "="*70)
print("FINAL CONCLUSIONS")
print("="*70)

summary = {
    'evaluation_date': pd.Timestamp.now().isoformat(),
    'bc_policy': str(BC_CHECKPOINT) if 'BC_CHECKPOINT' in globals() else 'Not evaluated',
    'residual_policy': str(RESIDUAL_CHECKPOINT),
    'num_rollouts': len(results.get('bc_only', {}).get('returns', [])),
}

if results['bc_only']['returns'] and results['bc_residual']['returns']:
    bc_returns = np.array(results['bc_only']['returns'])
    bc_res_returns = np.array(results['bc_residual']['returns'])
    
    summary.update({
        'bc_mean_return': float(np.mean(bc_returns)),
        'bc_std_return': float(np.std(bc_returns)),
        'bc_success_rate': results['bc_only']['successes'] / len(bc_returns),
        'bc_residual_mean_return': float(np.mean(bc_res_returns)),
        'bc_residual_std_return': float(np.std(bc_res_returns)),
        'bc_residual_success_rate': results['bc_residual']['successes'] / len(bc_res_returns),
        'improvement_pct': float((np.mean(bc_res_returns) - np.mean(bc_returns)) / (np.abs(np.mean(bc_returns)) + 1e-6) * 100),
    })
    
    print(f"\n✓ Successfully evaluated BC vs BC+Residual composition")
    print(f"  BC Mean Return:           {summary['bc_mean_return']:.4f} ± {summary['bc_std_return']:.4f}")
    print(f"  BC+Residual Mean Return:  {summary['bc_residual_mean_return']:.4f} ± {summary['bc_residual_std_return']:.4f}")
    print(f"  Improvement:              {summary['improvement_pct']:+.2f}%")
    
elif results['bc_only']['returns']:
    print(f"\n⚠ BC policy evaluated but BC+Residual composition failed or not available")
    bc_returns = np.array(results['bc_only']['returns'])
    summary['bc_mean_return'] = float(np.mean(bc_returns))
    summary['bc_std_return'] = float(np.std(bc_returns))
else:
    print(f"\n✗ Evaluation incomplete - no results to export")

print(f"\n✓ Evaluation complete. Results stored in 'results' dict.")
print(f"\nKey Findings:")
print(f"  - Composition formula: a_exec = clip(a_bc + residual)")
print(f"  - Residual policy requires 12-step history (first 12 steps use base policy only)")
print(f"  - Trajectory visualizations saved above")



FINAL CONCLUSIONS

⚠ BC policy evaluated but BC+Residual composition failed or not available

✓ Evaluation complete. Results stored in 'results' dict.

Key Findings:
  - Composition formula: a_exec = clip(a_bc + residual)
  - Residual policy requires 12-step history (first 12 steps use base policy only)
  - Trajectory visualizations saved above


## Next Steps: Full Rollout Evaluation

For more comprehensive policy evaluation with full environment interaction, use robomimic's evaluation scripts:

```bash
# Run 50 rollouts with video rendering
python run_trained_agent.py --agent /path/to/model.pth \
    --n_rollouts 50 --horizon 400 --seed 0 \
    --video_path /path/to/output.mp4 \
    --camera_names agentview robot0_eye_in_hand

# Write rollouts to HDF5 dataset
python run_trained_agent.py --agent /path/to/model.pth \
    --n_rollouts 50 --horizon 400 --seed 0 \
    --dataset_path /path/to/output.hdf5 --dataset_obs
```

**This notebook focused on:**
- Residual policy recovery quality (how well it predicts corrective actions)
- Comparison across different base policies (BC vs Diffusion)

**For full evaluation, also consider:**
- Task success rates in the environment
- Recovery success under different perturbation types and severities
- Comparison to baselines (no recovery, oracle recovery)
- Statistical significance testing

## Next Steps

**Current Benchmark**: BC vs BC+Residual composition

**Evaluation Framework Ready For**:
1. **Perturbed Trajectory Evaluation**: Apply synthetic perturbations (LATERAL_DRIFT, UNDERREACH_IDLE, etc.) and measure recovery
2. **Diffusion Policy Integration**: Extend to Diffusion+Residual composition
3. **Multi-Policy Comparison**: BC vs Diffusion vs BC+Residual vs Diffusion+Residual
4. **Trajectory Distribution Analysis**: Visualize policy diversity and robustness

**Architecture Notes**:
- Residual policy input: (B, 12, 74) = 12-step history of states (74-dim each)
- Residual policy output: (B, 30, 7) = 30-step horizon of action corrections
- Composition: `a_executed = clip(a_base + residuals[:, 0])`  (first timestep of residual window)
- History warmup: First 12 steps of episode use base policy only

**To Extend to Diffusion Policy**:
1. Create `ComposedDiffusionPolicy` wrapper similar to `ComposedBC_Residual`
2. Run parallel rollouts with `ComposedPolicy(diffusion, residual)`
3. Compare: BC+Residual vs Diffusion+Residual
